# Phase 3 v2 — Training Data Creation (Optimized)

**Key changes from v1:**
- Simplified prompt (shorter = less wasted tokens, more focus on prediction)
- Compact JSON response (no verbose notes — pure signal)
- Better numeric target derivation
- Messages-format JSONL compatible with SFTTrainer collation
- Data augmentation: 2 shuffled patch orderings per sample → 2× data


In [1]:
import os, sys, time

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/ImmunoPath"
DATA_DIR = f"{PROJECT_DIR}/data"
RESULTS_DIR = f"{PROJECT_DIR}/results"

os.makedirs(f"{DATA_DIR}/training_v2", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/phase3_v2", exist_ok=True)

print("Directories ready")

Mounted at /content/drive
Directories ready


## Install Dependencies

In [2]:
import subprocess
subprocess.run(["pip", "install", "-q", "pandas", "numpy", "tqdm", "scikit-learn"], check=True)

import json
import glob
import random
import math
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

print("Dependencies loaded")

Dependencies loaded


## Configuration

In [3]:
MAX_PATCHES_PER_SAMPLE = 8     # MedGemma multi-image capacity
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
RANDOM_SEED = 42
N_AUGMENTATIONS = 2            # Patch-order shuffles per sample (data augmentation)

CANCER_TYPES = {
    "TCGA-LUAD": "lung adenocarcinoma",
    "TCGA-LUSC": "lung squamous cell carcinoma",
}

# Paths
PATCHES_DIR     = f"{DATA_DIR}/processed/patches"
SIGNATURES_PATH = f"{DATA_DIR}/signatures/immune_signatures.csv"
METADATA_PATH   = f"{DATA_DIR}/raw/nsclc_metadata.csv"
BAGAEV_PATH     = f"{DATA_DIR}/labels/bagaev_tme/annotation.tsv"
THORSSON_DIR    = f"{DATA_DIR}/labels/thorsson_panimmune"
CBIO_DIR        = f"{DATA_DIR}/labels/cbioportal"
OUTPUT_DIR      = f"{DATA_DIR}/training_v2"

print("Config set")
print(f"  Augmentations per sample: {N_AUGMENTATIONS}")
print(f"  Output: {OUTPUT_DIR}")

Config set
  Augmentations per sample: 2
  Output: /content/drive/MyDrive/ImmunoPath/data/training_v2


## Load Immune Signatures (Phase 2 output)

In [4]:
signatures_df = pd.read_csv(SIGNATURES_PATH, index_col=0)
signatures_df.index.name = "sample_id"
print(f"Immune signatures: {signatures_df.shape[0]} patients × {signatures_df.shape[1]} columns")
print(f"   Columns: {list(signatures_df.columns)}")
print(f"\n   CD274 distribution:")
if "cd274_expression" in signatures_df.columns:
    print(signatures_df["cd274_expression"].value_counts().to_string())
print(f"\n   Immune phenotype distribution:")
if "immune_phenotype" in signatures_df.columns:
    print(signatures_df["immune_phenotype"].value_counts().to_string())

Immune signatures: 950 patients × 9 columns
   Columns: ['cd8_signature', 'ifng_signature', 'til_signature', 'pdl1_related', 'exhaustion_signature', 'cd274_log2_tpm', 'cd274_expression', 'immune_phenotype', 'immune_score']

   CD274 distribution:
cd274_expression
high    475
low     475

   Immune phenotype distribution:
immune_phenotype
desert      475
inflamed    383
excluded     92


## Load Slide Metadata + Scan Patch Directories (Parallel)

In [5]:
metadata_df = pd.read_csv(METADATA_PATH)
print(f"Slide metadata: {len(metadata_df)} rows")
metadata_df["slide_id"] = metadata_df["file_name"].str.replace(".svs", "", regex=False)

def scan_slide_patches(slide_dir: str) -> dict:
    """Scan a single slide patch directory. Prefers diversity-selected patches."""
    slide_id = os.path.basename(slide_dir)
    all_patches = sorted(glob.glob(os.path.join(slide_dir, "*.jpg")))
    if not all_patches:
        return None

    # Prefer diversity-selected patches
    selected_path = os.path.join(slide_dir, f"{slide_id}_selected_8.json")
    if os.path.exists(selected_path):
        try:
            with open(selected_path) as f:
                sel = json.load(f)
            selected_names = sel.get("selected", [])
            if selected_names:
                patches = [os.path.join(slide_dir, name) for name in selected_names
                           if os.path.exists(os.path.join(slide_dir, name))]
                method = sel.get("method", "kmeans") if patches else "fallback"
                if not patches:
                    patches = all_patches[:MAX_PATCHES_PER_SAMPLE]
                    method = "fallback_sorted"
            else:
                patches = all_patches[:MAX_PATCHES_PER_SAMPLE]
                method = "fallback_sorted"
        except Exception:
            patches = all_patches[:MAX_PATCHES_PER_SAMPLE]
            method = "fallback_sorted"
    else:
        patches = all_patches[:MAX_PATCHES_PER_SAMPLE]
        method = "fallback_sorted"

    return {
        "slide_id": slide_id,
        "n_patches": len(patches),
        "n_total_patches": len(all_patches),
        "patch_paths": patches,
        "selection_method": method,
    }

slide_dirs = [
    os.path.join(PATCHES_DIR, d)
    for d in os.listdir(PATCHES_DIR)
    if os.path.isdir(os.path.join(PATCHES_DIR, d))
] if os.path.exists(PATCHES_DIR) else []

print(f"Scanning {len(slide_dirs)} patch directories...")
patch_data = {}
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(scan_slide_patches, d): d for d in slide_dirs}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Scanning"):
        result = future.result()
        if result is not None:
            patch_data[result["slide_id"]] = result

print(f"Found patches for {len(patch_data)} slides")
total_patches = sum(d["n_patches"] for d in patch_data.values())
print(f"   Total patches (selected): {total_patches}")

Slide metadata: 950 rows
Scanning 1060 patch directories...


Scanning:   0%|          | 0/1060 [00:00<?, ?it/s]

Found patches for 949 slides
   Total patches (selected): 7574


## Load External Labels (Bagaev TME + MSI)

In [6]:
# --- Bagaev TME subtypes ---
bagaev_tme = {}
if os.path.exists(BAGAEV_PATH):
    try:
        bagaev_df = pd.read_csv(BAGAEV_PATH, sep='\t')
        print(f"Bagaev TME: {len(bagaev_df)} rows")
        id_col = bagaev_df.columns[0]
        mfp_col = "MFP" if "MFP" in bagaev_df.columns else None
        if mfp_col:
            bagaev_df[mfp_col] = bagaev_df[mfp_col].str.replace("IE_F", "IE/F", regex=False)
            bagaev_df["patient_id"] = bagaev_df[id_col].astype(str).str[:12]
            lung_bagaev = bagaev_df[
                bagaev_df["patient_id"].str.startswith("TCGA-")
            ].drop_duplicates(subset="patient_id", keep="first")
            bagaev_tme = dict(zip(lung_bagaev["patient_id"], lung_bagaev[mfp_col]))
            print(f"   Matched: {len(bagaev_tme)} | TME: {Counter(bagaev_tme.values())}")
    except Exception as e:
        print(f"Error loading Bagaev TME: {e}")

# --- MSI Status (multi-source priority merge) ---
msi_labels = {}
msi_source_counts = {"bagaev": 0, "cbioportal": 0, "thorsson_c6": 0}

# Source 1: Bagaev MSI column
if 'bagaev_df' in dir() and 'MSI' in bagaev_df.columns:
    bagaev_df["_pid"] = bagaev_df[bagaev_df.columns[0]].astype(str).str[:12]
    for _, row in bagaev_df.iterrows():
        pid = row["_pid"]
        msi_val = str(row["MSI"]).strip().upper()
        if msi_val in ("MSI-H", "MSI_H", "MSI"):
            msi_labels[pid] = "MSI-H"
            msi_source_counts["bagaev"] += 1
        elif msi_val in ("MSS", "MSI-L", "MSI_L", "STABLE"):
            if pid not in msi_labels:
                msi_labels[pid] = "MSS"
        elif msi_val not in ("NAN", "", "NA", "NONE"):
            if pid not in msi_labels:
                msi_labels[pid] = "MSS"
    bagaev_df.drop(columns=["_pid"], inplace=True, errors="ignore")
    print(f"Bagaev MSI: {sum(1 for v in msi_labels.values() if v == 'MSI-H')} MSI-H")

# Source 2: cBioPortal MSIsensor/MANTIS scores
for study in ["luad_tcga_pan_can_atlas_2018", "lusc_tcga_pan_can_atlas_2018"]:
    clinical_path = os.path.join(CBIO_DIR, study, "data_clinical_sample.txt")
    if not os.path.exists(clinical_path):
        clinical_path = os.path.join(CBIO_DIR, study, "data_clinical_patient.txt")
    if os.path.exists(clinical_path):
        try:
            cdf = pd.read_csv(clinical_path, sep='\t', comment='#')
            id_col_c = cdf.columns[0]
            for _, row in cdf.iterrows():
                pid = str(row[id_col_c])[:12]
                if pid in msi_labels:
                    continue
                msisensor = mantis = None
                for c in cdf.columns:
                    cl = c.lower()
                    if "msi_sensor" in cl or "msisensor" in cl:
                        try: msisensor = float(row[c])
                        except (ValueError, TypeError): pass
                    elif "mantis" in cl:
                        try: mantis = float(row[c])
                        except (ValueError, TypeError): pass
                if msisensor is not None and msisensor >= 3.5:
                    msi_labels[pid] = "MSI-H"
                    msi_source_counts["cbioportal"] += 1
                elif mantis is not None and mantis >= 0.4:
                    msi_labels[pid] = "MSI-H"
                    msi_source_counts["cbioportal"] += 1
                elif msisensor is not None or mantis is not None:
                    msi_labels[pid] = "MSS"
        except Exception as e:
            print(f"   cBioPortal {study}: {e}")

print(f"cBioPortal: +{msi_source_counts['cbioportal']} MSI-H")

# Source 3: Thorsson C6 fallback
subtypes_path = os.path.join(THORSSON_DIR, "tcga_subtypes.tsv")
if os.path.exists(subtypes_path):
    try:
        subtypes_df = pd.read_csv(subtypes_path, sep='\t')
        for col in subtypes_df.columns:
            if "immune" in col.lower() and "subtype" in col.lower():
                id_col_t = subtypes_df.columns[0]
                subtypes_df["patient_id"] = subtypes_df[id_col_t].astype(str).str[:12]
                for _, row in subtypes_df.iterrows():
                    pid = row["patient_id"]
                    if pid in msi_labels:
                        continue
                    if str(row[col]) == "C6":
                        msi_labels[pid] = "MSI-H"
                        msi_source_counts["thorsson_c6"] += 1
                    else:
                        msi_labels[pid] = "MSS"
                break
    except Exception as e:
        print(f"   Thorsson: {e}")

total_msih = sum(1 for v in msi_labels.values() if v == "MSI-H")
total_mss = sum(1 for v in msi_labels.values() if v == "MSS")
print(f"\nMSI combined: {len(msi_labels)} patients | MSI-H: {total_msih} | MSS: {total_mss}")

Bagaev TME: 8024 rows
   Matched: 8024 | TME: Counter({'D': 3133, 'IE': 1912, 'F': 1718, 'IE/F': 1261})
Bagaev MSI: 240 MSI-H
cBioPortal: +1 MSI-H

MSI combined: 7707 patients | MSI-H: 241 | MSS: 7466


## Join All Data Sources → Matched Samples

In [7]:
print("=" * 40)
print("JOINING DATA SOURCES")
print("=" * 40)

patient_to_slides = defaultdict(list)
for _, row in metadata_df.iterrows():
    patient_to_slides[row["submitter_id"]].append(row["slide_id"])
patient_to_slide = {pid: sids[0] for pid, sids in patient_to_slides.items()}
slide_to_project = dict(zip(metadata_df["slide_id"], metadata_df["project_id"]))

matched_samples = []
unmatched_reasons = Counter()

for sample_id in signatures_df.index:
    patient_id = sample_id[:12]
    slide_id = patient_to_slide.get(patient_id)
    if slide_id is None:
        for sid in patient_to_slides.get(patient_id, []):
            if sid in patch_data:
                slide_id = sid
                break
    if slide_id is None:
        unmatched_reasons["no_slide_mapping"] += 1
        continue
    if slide_id not in patch_data:
        unmatched_reasons["no_patches"] += 1
        continue
    pdata = patch_data[slide_id]
    if pdata["n_patches"] == 0:
        unmatched_reasons["zero_patches"] += 1
        continue

    project_id = slide_to_project.get(slide_id, "TCGA-LUAD")
    cancer_type = CANCER_TYPES.get(project_id, "lung cancer")
    tme_subtype = bagaev_tme.get(patient_id, "unknown")
    msi_status = msi_labels.get(patient_id, "unknown")
    sig = signatures_df.loc[sample_id].to_dict()

    matched_samples.append({
        "sample_id": sample_id,
        "patient_id": patient_id,
        "slide_id": slide_id,
        "project_id": project_id,
        "cancer_type": cancer_type,
        "patch_paths": pdata["patch_paths"],
        "n_patches": pdata["n_patches"],
        "tme_subtype": tme_subtype,
        "msi_status": msi_status,
        "immune_signature": sig,
    })

print(f"\nMatched samples: {len(matched_samples)}")
print(f"   Unmatched: {dict(unmatched_reasons)}")
projects = Counter(s["project_id"] for s in matched_samples)
tme_dist = Counter(s["tme_subtype"] for s in matched_samples)
msi_dist = Counter(s["msi_status"] for s in matched_samples)
print(f"   Projects: {dict(projects)}")
print(f"   TME: {dict(tme_dist)}")
print(f"   MSI: {dict(msi_dist)}")

JOINING DATA SOURCES

Matched samples: 939
   Unmatched: {'no_patches': 11}
   Projects: {'TCGA-LUSC': 469, 'TCGA-LUAD': 470}
   TME: {'IE': 220, 'F': 211, 'unknown': 25, 'IE/F': 153, 'D': 330}
   MSI: {'MSS': 930, 'MSI-H': 7, 'unknown': 2}


## Build Prompt + Response (v2 — Compact, Signal-Dense)

**Key changes:**
- Shorter prompt → less wasted tokens (entire sequence is in loss now)
- JSON response is compact — no verbose notes, just prediction values
- Added `cd8_infiltration` and `til_density` as string labels the model can learn more easily
- All numeric values are well-bounded (0-1 range)

In [8]:

# v2 PROMPT — compact, direct, classification-focused

PROMPT_TEMPLATE = """Analyze these H&E histopathology patches from a {cancer_type} biopsy.

Predict the tumor immune microenvironment as JSON:
- cd274_expression: "high" or "low" (RNA proxy, not IHC)
- msi_status: "MSI-H" or "MSS"
- tme_subtype: "IE", "IE/F", "F", or "D"
- immune_phenotype: "inflamed", "excluded", or "desert"
- cd8_infiltration: "low", "moderate", or "high"
- til_density: "low", "moderate", or "high"
- til_fraction: float 0.0-1.0
- immune_score: float 0.0-1.0

Output ONLY the JSON object."""


def categorize_score(score: float) -> str:
    """Convert z-score to categorical label."""
    if pd.isna(score):
        return "unknown"
    score = float(score)
    if score < -0.5:
        return "low"
    elif score < 0.5:
        return "moderate"
    else:
        return "high"


def zscore_to_fraction(z: float) -> float:
    """Sigmoid: z-score → 0-1 fraction."""
    try:
        z = float(z)
        if math.isnan(z):
            return 0.5
        return round(1.0 / (1.0 + math.exp(-z)), 3)
    except (ValueError, TypeError, OverflowError):
        return 0.5


def build_target_response_v2(sample: dict) -> str:
    """Build compact JSON response — pure prediction values, no notes."""
    sig = sample["immune_signature"]

    til_sig_raw = sig.get("til_signature", 0.0)
    til_fraction = zscore_to_fraction(til_sig_raw if not pd.isna(til_sig_raw) else 0.0)
    til_density = categorize_score(float(til_sig_raw) if not pd.isna(til_sig_raw) else 0.0)

    cd8_sig = sig.get("cd8_signature", 0.0)
    cd8_infiltration = categorize_score(float(cd8_sig) if not pd.isna(cd8_sig) else 0.0)

    immune_score = round(float(sig.get("immune_score", 0.5)), 3)

    response = {
        "cd274_expression": sig.get("cd274_expression", "unknown"),
        "msi_status": sample.get("msi_status", "unknown"),
        "tme_subtype": sample.get("tme_subtype", "unknown"),
        "immune_phenotype": sig.get("immune_phenotype", "unknown"),
        "cd8_infiltration": cd8_infiltration,
        "til_density": til_density,
        "til_fraction": til_fraction,
        "immune_score": immune_score,
    }
    # Compact JSON — single-line, no indent (saves tokens)
    return json.dumps(response)



# Build training samples with data augmentation

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

training_samples = []
for sample in tqdm(matched_samples, desc="Building samples"):
    patches = sample["patch_paths"][:MAX_PATCHES_PER_SAMPLE]
    prompt = PROMPT_TEMPLATE.format(cancer_type=sample["cancer_type"])
    response = build_target_response_v2(sample)

    # Create N_AUGMENTATIONS versions with shuffled patch order
    for aug_idx in range(N_AUGMENTATIONS):
        shuffled_patches = patches.copy()
        if aug_idx > 0:
            random.shuffle(shuffled_patches)

        training_samples.append({
            "sample_id": sample["sample_id"],
            "patient_id": sample["patient_id"],
            "slide_id": sample["slide_id"],
            "project_id": sample["project_id"],
            "cancer_type": sample["cancer_type"],
            "patch_paths": shuffled_patches,
            "n_patches": len(shuffled_patches),
            "prompt": prompt,
            "response": response,
            "aug_idx": aug_idx,
            "immune_signature": sample["immune_signature"],
        })

print(f"\nBuilt {len(training_samples)} training samples ({N_AUGMENTATIONS}x augmentation)")
print(f"   Unique patients: {len(set(s['patient_id'] for s in training_samples))}")

# Show a sample prompt + response
if training_samples:
    s = training_samples[0]
    print(f"\n--- Sample prompt ---")
    print(s["prompt"][:300])
    print(f"\n--- Sample response ---")
    print(s["response"])

Building samples:   0%|          | 0/939 [00:00<?, ?it/s]


Built 1878 training samples (2x augmentation)
   Unique patients: 939

--- Sample prompt ---
Analyze these H&E histopathology patches from a lung squamous cell carcinoma biopsy.

Predict the tumor immune microenvironment as JSON:
- cd274_expression: "high" or "low" (RNA proxy, not IHC)
- msi_status: "MSI-H" or "MSS"
- tme_subtype: "IE", "IE/F", "F", or "D"
- immune_phenotype: "inflamed", "e

--- Sample response ---
{"cd274_expression": "high", "msi_status": "MSS", "tme_subtype": "IE", "immune_phenotype": "inflamed", "cd8_infiltration": "moderate", "til_density": "high", "til_fraction": 0.675, "immune_score": 0.626}


## Patient-Level Split (Stratified)

Split by PATIENT ID first, then augmented copies stay together.

In [9]:
from sklearn.model_selection import train_test_split

def patient_level_split_v2(samples, train_ratio=0.80, val_ratio=0.10, test_ratio=0.10, seed=42):
    """Split by patient ID. Augmented copies of same patient go to same split."""
    random.seed(seed)
    np.random.seed(seed)

    # Group by patient
    patient_to_samples = defaultdict(list)
    for s in samples:
        patient_to_samples[s["patient_id"]].append(s)

    patient_ids = sorted(patient_to_samples.keys())
    n = len(patient_ids)

    # Stratify by cancer_type + cd274
    strat_labels = []
    for pid in patient_ids:
        s = patient_to_samples[pid][0]
        sig = s.get("immune_signature", {})
        label = f"{s['project_id']}_{sig.get('cd274_expression', 'unk')}"
        strat_labels.append(label)

    # Handle rare labels
    label_counts = Counter(strat_labels)
    safe_labels = [lbl if label_counts[lbl] >= 3 else "RARE" for lbl in strat_labels]

    try:
        rest_ratio = val_ratio + test_ratio
        train_idx, rest_idx = train_test_split(
            range(n), test_size=rest_ratio, stratify=safe_labels, random_state=seed
        )
        rest_labels = [safe_labels[i] for i in rest_idx]
        val_frac = val_ratio / rest_ratio
        val_idx, test_idx = train_test_split(
            rest_idx, test_size=1 - val_frac, stratify=rest_labels, random_state=seed
        )
        train_pids = [patient_ids[i] for i in train_idx]
        val_pids = [patient_ids[i] for i in val_idx]
        test_pids = [patient_ids[i] for i in test_idx]
        print("(Stratified split successful)")
    except ValueError as e:
        print(f"Stratified split failed ({e}), random fallback")
        random.shuffle(patient_ids)
        te = int(n * train_ratio)
        ve = te + int(n * val_ratio)
        train_pids, val_pids, test_pids = patient_ids[:te], patient_ids[te:ve], patient_ids[ve:]

    # Collect samples — augmented copies go with their patient
    train = [s for pid in train_pids for s in patient_to_samples[pid]]
    val_all = [s for pid in val_pids for s in patient_to_samples[pid]]
    test_all = [s for pid in test_pids for s in patient_to_samples[pid]]

    # For val/test: keep only aug_idx=0 (no augmentation for evaluation)
    val = [s for s in val_all if s.get("aug_idx", 0) == 0]
    test = [s for s in test_all if s.get("aug_idx", 0) == 0]

    print(f"\nPatient-level split (seed={seed}):")
    print(f"   Train: {len(set(s['patient_id'] for s in train))} patients → {len(train)} samples (with augmentation)")
    print(f"   Val:   {len(val_pids)} patients → {len(val)} samples (no augmentation)")
    print(f"   Test:  {len(test_pids)} patients → {len(test)} samples (no augmentation)")

    # Verify no leakage
    train_set = set(s["patient_id"] for s in train)
    val_set = set(s["patient_id"] for s in val)
    test_set = set(s["patient_id"] for s in test)
    assert len(train_set & val_set) == 0, "LEAKAGE: train ∩ val"
    assert len(train_set & test_set) == 0, "LEAKAGE: train ∩ test"
    assert len(val_set & test_set) == 0, "LEAKAGE: val ∩ test"
    print("   No patient leakage ✓")

    return train, val, test

train_samples, val_samples, test_samples = patient_level_split_v2(
    training_samples, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, RANDOM_SEED
)

(Stratified split successful)

Patient-level split (seed=42):
   Train: 751 patients → 1502 samples (with augmentation)
   Val:   94 patients → 94 samples (no augmentation)
   Test:  94 patients → 94 samples (no augmentation)
   No patient leakage ✓


## Write JSONL Files (v2 format — messages-compatible)

Each record stores the data needed for SFTTrainer collation:
- `patch_paths`: list of image file paths
- `prompt`: the user-turn text
- `response`: the assistant-turn text (compact JSON)

In [10]:
def write_jsonl(samples: list, output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', buffering=1024*1024) as f:
        for sample in samples:
            record = {
                "sample_id": sample["sample_id"],
                "patient_id": sample["patient_id"],
                "slide_id": sample["slide_id"],
                "project_id": sample["project_id"],
                "cancer_type": sample["cancer_type"],
                "patch_paths": sample["patch_paths"],
                "prompt": sample["prompt"],
                "response": sample["response"],
            }
            f.write(json.dumps(record) + '\n')
    print(f"Written {len(samples)} samples to {output_path}")

write_jsonl(train_samples, f"{OUTPUT_DIR}/train.jsonl")
write_jsonl(val_samples, f"{OUTPUT_DIR}/val.jsonl")
write_jsonl(test_samples, f"{OUTPUT_DIR}/test.jsonl")

# Also save split metadata
with open(f"{OUTPUT_DIR}/split_metadata.json", 'w') as f:
    json.dump({
        "train_patients": sorted(set(s["patient_id"] for s in train_samples)),
        "val_patients": sorted(set(s["patient_id"] for s in val_samples)),
        "test_patients": sorted(set(s["patient_id"] for s in test_samples)),
        "total_unique_patients": len(set(s["patient_id"] for s in training_samples)),
        "augmentation_factor": N_AUGMENTATIONS,
        "seed": RANDOM_SEED,
    }, f, indent=2)

print(f"\nAll JSONL files written to {OUTPUT_DIR}/")

Written 1502 samples to /content/drive/MyDrive/ImmunoPath/data/training_v2/train.jsonl
Written 94 samples to /content/drive/MyDrive/ImmunoPath/data/training_v2/val.jsonl
Written 94 samples to /content/drive/MyDrive/ImmunoPath/data/training_v2/test.jsonl

All JSONL files written to /content/drive/MyDrive/ImmunoPath/data/training_v2/


## Verify Dataset Integrity

In [11]:
def verify_jsonl(path: str) -> dict:
    stats = {"total": 0, "valid_json": 0, "valid_paths": 0, "missing_patches": 0}
    with open(path) as f:
        for line in f:
            stats["total"] += 1
            try:
                record = json.loads(line)
                stats["valid_json"] += 1
                all_exist = all(os.path.exists(p) for p in record["patch_paths"])
                if all_exist:
                    stats["valid_paths"] += 1
                else:
                    stats["missing_patches"] += sum(
                        1 for p in record["patch_paths"] if not os.path.exists(p)
                    )
                # Verify response is valid compact JSON
                resp = json.loads(record["response"])
                assert "cd274_expression" in resp, "Missing cd274_expression"
                assert "immune_score" in resp, "Missing immune_score"
                assert "tme_subtype" in resp, "Missing tme_subtype"
            except (json.JSONDecodeError, KeyError, AssertionError) as e:
                print(f"Invalid record: {e}")
    return stats

print("Verifying datasets...")
for split_name in ["train", "val", "test"]:
    path = f"{OUTPUT_DIR}/{split_name}.jsonl"
    if os.path.exists(path):
        stats = verify_jsonl(path)
        print(f"\n   {split_name}.jsonl:")
        print(f"      Records: {stats['total']}")
        print(f"      Valid JSON: {stats['valid_json']}")
        print(f"      Valid paths: {stats['valid_paths']}")
        if stats["missing_patches"] > 0:
            print(f"      ⚠️  Missing patches: {stats['missing_patches']}")

Verifying datasets...

   train.jsonl:
      Records: 1502
      Valid JSON: 1502
      Valid paths: 1502

   val.jsonl:
      Records: 94
      Valid JSON: 94
      Valid paths: 94

   test.jsonl:
      Records: 94
      Valid JSON: 94
      Valid paths: 94


## Label Distribution Analysis

In [12]:
def analyze_split(samples, name):
    if not samples:
        print(f"\n{name}: EMPTY")
        return
    print(f"\n{name} ({len(samples)} samples):")
    cd274 = Counter(json.loads(s["response"]).get("cd274_expression") for s in samples)
    tme = Counter(json.loads(s["response"]).get("tme_subtype") for s in samples)
    msi = Counter(json.loads(s["response"]).get("msi_status") for s in samples)
    pheno = Counter(json.loads(s["response"]).get("immune_phenotype") for s in samples)
    print(f"   CD274: {dict(cd274)}")
    print(f"   TME:   {dict(tme)}")
    print(f"   MSI:   {dict(msi)}")
    print(f"   Phenotype: {dict(pheno)}")

analyze_split(train_samples, "TRAIN")
analyze_split(val_samples, "VAL")
analyze_split(test_samples, "TEST")


TRAIN (1502 samples):
   CD274: {'low': 752, 'high': 750}
   TME:   {'D': 522, 'unknown': 42, 'F': 336, 'IE/F': 248, 'IE': 354}
   MSI:   {'MSS': 1490, 'unknown': 4, 'MSI-H': 8}
   Phenotype: {'desert': 748, 'inflamed': 616, 'excluded': 138}

VAL (94 samples):
   CD274: {'high': 47, 'low': 47}
   TME:   {'F': 19, 'D': 36, 'IE': 18, 'IE/F': 18, 'unknown': 3}
   MSI:   {'MSS': 93, 'MSI-H': 1}
   Phenotype: {'desert': 46, 'inflamed': 34, 'excluded': 14}

TEST (94 samples):
   CD274: {'high': 47, 'low': 47}
   TME:   {'F': 24, 'IE': 25, 'D': 33, 'IE/F': 11, 'unknown': 1}
   MSI:   {'MSS': 92, 'MSI-H': 2}
   Phenotype: {'desert': 49, 'inflamed': 37, 'excluded': 8}


## Phase 3 v2 Summary

In [14]:
report = {
    "phase": "3_v2",
    "timestamp": datetime.now().isoformat(),
    "total_matched_patients": len(matched_samples),
    "augmentation_factor": N_AUGMENTATIONS,
    "train_samples": len(train_samples),
    "val_samples": len(val_samples),
    "test_samples": len(test_samples),
    "output_dir": OUTPUT_DIR,
    "changes_from_v1": [
        "Compact JSON response (no verbose notes)",
        "Shorter prompt (less token waste)",
        f"{N_AUGMENTATIONS}x patch-order augmentation",
        "Val/test have no augmentation",
        "Single-line JSON (saves ~50 tokens per sample)",
    ],
}
with open(f"{RESULTS_DIR}/phase3_v2/report.json", 'w') as f:
    json.dump(report, f, indent=2)

print("=" * 60)
print("PHASE 3 v2 — TRAINING DATA CREATION COMPLETE")
print("=" * 60)
print(f"  Matched patients:   {len(matched_samples)}")
print(f"  Augmentation:       {N_AUGMENTATIONS}x")
print(f"  Train: {len(train_samples)} samples")
print(f"  Val:   {len(val_samples)} samples")
print(f"  Test:  {len(test_samples)} samples")
print(f"  Output: {OUTPUT_DIR}/")


PHASE 3 v2 — TRAINING DATA CREATION COMPLETE
  Matched patients:   939
  Augmentation:       2x
  Train: 1502 samples
  Val:   94 samples
  Test:  94 samples
  Output: /content/drive/MyDrive/ImmunoPath/data/training_v2/
